<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/_Kids_Elementary_Number_Places_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploding Dots: Educational Animation Overview

## Educational Objectives
This animation demonstrates the "Exploding Dots" model, a visual representation of the base-10 place value system. It is designed for elementary education to help students visualize:
1. Accumulation: How single units (1s) build up.
2. Regrouping/Carrying: The physical 'explosion' of 10 units in one column to create 1 unit in the next column to the left.
3. Positional Value: The distinct roles of the 1s, 10s, and 100s columns.

## Visual Design and Accessibility
- Background: Soft cream (#FDF8E7) to reduce eye strain.
- Color Coding: High-contrast colors are used to distinguish place values (Red for 1s, Blue for 10s, Green for 100s).
- UI Elements: Thick 4px rounded outlines provide clear visual boundaries for place value containers.
- Pacing: The animation includes intentional pauses during 'explosions' to allow for verbal explanation or student observation.

## Technical Specifications
- Format: MP4
- Frame Rate: 30 FPS
- Resolution: 1200x800 (100 DPI)
- Rendering: Matplotlib FFMpegWriter
- Attribution: © Mugambi Ndwiga / @craftsandengineering
- Source Code: [GitHub Repository](https://github.com/zombimann/Mathematical-video-animations-and-visualization)

## Usage Instructions
Educators can use this video to introduce multi-digit addition or the concept of carrying. Parents can use it as a visual aid when students are struggling with the abstract nature of 'moving the one' in standard algorithms.

In [ ]:
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
GitHub: https://github.com/zombimann/Mathematical-video-animations-and-visualization
Level: Elementary
Concept: Exploding Dots (Place Value)
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Rectangle, FancyBboxPatch
import matplotlib.patheffects as path_effects

# --- PARAMETERS ---
FPS = 30
DOT_SPEED = 10 # Frames per dot
EXPLOSION_PAUSE = 25 # Frames to wait during explosion
CYCLES_TO_SHOW = 12 # Number of carry-overs to animate
END_CARD_DURATION = FPS * 3 # 3 seconds

# Calculate frames needed to see 100s column activity
frames_per_cycle = (10 * DOT_SPEED + EXPLOSION_PAUSE)
TOTAL_FRAMES = (frames_per_cycle * CYCLES_TO_SHOW) + END_CARD_DURATION

BG_COLOR = "#FDF8E7"
GRID_COLOR = "#D3D3D3"
COLORS = {"1s": "#FF6B6B", "10s": "#4D96FF", "100s": "#6BCB77"}
TEXT_COLOR = "#2D2D2D"
WATERMARK = "© Mugambi Ndwiga / @craftsandengineering"

fig, ax = plt.subplots(figsize=(12, 8), dpi=100)
fig.patch.set_facecolor(BG_COLOR)

def setup_ui(ax):
    ax.clear()
    ax.set_facecolor(BG_COLOR)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.axis('off')
    # Subtle Grid
    for i in range(0, 101, 10):
        ax.axvline(i, color=GRID_COLOR, alpha=0.08)
        ax.axhline(i, color=GRID_COLOR, alpha=0.08)

    # Place Value Boxes
    cols = [("100s", 15), ("10s", 40), ("1s", 65)]
    for label, x in cols:
        rect = FancyBboxPatch((x, 20), 20, 50, boxstyle="round,pad=2",
                              linewidth=4, edgecolor=TEXT_COLOR, facecolor='none')
        ax.add_patch(rect)
        txt = ax.text(x + 10, 12, label, ha='center', fontsize=36, fontweight='bold', color=TEXT_COLOR)
        txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])

    # Header and Watermark
    rect_l = FancyBboxPatch((65, 88), 30, 7, boxstyle="round,pad=1", facecolor='#4D96FF', alpha=0.8)
    ax.add_patch(rect_l)
    ax.text(80, 91.5, "For Kids: Elementary", color='white', ha='center', va='center', fontsize=18, fontweight='bold')
    ax.text(98, 2, WATERMARK, ha='right', va='bottom', fontsize=12, alpha=0.6, color=TEXT_COLOR)

def get_pos(col_idx, count):
    base_x = [15, 40, 65][col_idx]
    return base_x + 5 + (count % 2) * 10, 25 + (count // 2) * 8

def animate(i):
    # Handle Closing Title Card
    if i >= TOTAL_FRAMES - END_CARD_DURATION:
        ax.clear()
        ax.set_facecolor('#4D96FF')
        ax.text(50, 55, "Made by Mugambi Ndwiga", color='white', ha='center', fontsize=36, fontweight='bold')
        ax.text(50, 45, "@craftsandengineering", color='white', ha='center', fontsize=24)
        ax.axis('off')
        return

    setup_ui(ax)

    cycle_idx = i // frames_per_cycle
    intra_cycle_frame = i % frames_per_cycle

    is_exploding_1s = intra_cycle_frame >= (10 * DOT_SPEED)

    # Logical counts for place value carry-over
    tens_logic = (cycle_idx % 10) + (1 if is_exploding_1s else 0)
    hundreds_logic = (cycle_idx // 10)

    # Transition logic for 10s to 100s
    is_exploding_10s = (tens_logic >= 10 and is_exploding_1s)
    if is_exploding_10s:
        tens_to_show = 10
        hundreds_count = hundreds_logic + 1
    else:
        tens_to_show = tens_logic % 10
        hundreds_count = hundreds_logic

    # Draw Hundreds Column
    for d in range(hundreds_count):
        ax.add_patch(Circle(get_pos(0, d), 3.5, facecolor=COLORS["100s"], edgecolor=TEXT_COLOR, linewidth=3))

    # Draw Tens Column
    if is_exploding_10s:
        for d in range(10):
            ax.add_patch(Circle(get_pos(1, d), 3.5, facecolor=COLORS["10s"], edgecolor=TEXT_COLOR, linewidth=3, alpha=0.3))
        ann = ax.text(35, 75, "10 explode to 100!", fontsize=24, color=TEXT_COLOR, ha='center', fontweight='bold')
        ann.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])
    else:
        for d in range(tens_to_show):
            ax.add_patch(Circle(get_pos(1, d), 3.5, facecolor=COLORS["10s"], edgecolor=TEXT_COLOR, linewidth=3))

    # Draw Ones Column
    if not is_exploding_1s:
        ones_to_draw = intra_cycle_frame // DOT_SPEED
        for d in range(ones_to_draw):
            ax.add_patch(Circle(get_pos(2, d), 3.5, facecolor=COLORS["1s"], edgecolor=TEXT_COLOR, linewidth=3))
    else:
        for d in range(10):
            ax.add_patch(Circle(get_pos(2, d), 3.5, facecolor=COLORS["1s"], edgecolor=TEXT_COLOR, linewidth=3, alpha=0.3))
        if not is_exploding_10s:
            ann = ax.text(75, 75, "10 explode!", fontsize=24, color=TEXT_COLOR, ha='center', fontweight='bold')
            ann.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])

ani = FuncAnimation(fig, animate, frames=TOTAL_FRAMES, interval=1000/FPS)
video_path = 'exploding_dots_edu.mp4'
writer = FFMpegWriter(fps=FPS, bitrate=1800)
ani.save(video_path, writer=writer)
plt.close()

from IPython.display import Video, display, HTML
display(Video(video_path, embed=True))
display(HTML(f'<a href="{video_path}" download>Click here to download the video</a>'))

In [ ]:
# Installation of necessary dependencies
!apt-get install -y ffmpeg
!pip install numpy matplotlib

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
